# Order flow imbalance and price impact

This notebook replicates the central result of

> Cont, R., Kukanov, A. and Stoikov, S. (2014). *The price impact of order book events.*
> Journal of Financial Econometrics, 12(1), 47–88.

using best bid/offer data reconstructed from the NASDAQ ITCH 5.0 sample file. Each observation
is a book event, meaning a change in the price or size of the best bid or ask. This is the
sampling scheme the paper's order flow imbalance measure is defined on.

**Hypothesis.** For SPY, QQQ, AMD, INTC, GOOGL, over 10-second intervals in regular trading
hours, the mid-price change in ticks satisfies $\Delta P_k = \alpha + \beta\,\mathrm{OFI}_k + \varepsilon_k$
with $\beta > 0$ statistically significant and $R^2$ of the same order as the paper (~50–70%
for liquid names). Falsified if $\beta$ is insignificant/negative, or $R^2$ is near zero, or the
relation is clearly non-linear.

**Order flow imbalance.** Write $P^b_n$ and $q^b_n$ for the best bid price and the number of
shares quoted at it immediately after event $n$, and $P^a_n$ and $q^a_n$ for the best ask price
and its quoted size. Each event contributes (paper eq. 10)

$$e_n = \mathbb{1}_{\{P^b_n \ge P^b_{n-1}\}} q^b_n - \mathbb{1}_{\{P^b_n \le P^b_{n-1}\}} q^b_{n-1}
      - \mathbb{1}_{\{P^a_n \le P^a_{n-1}\}} q^a_n + \mathbb{1}_{\{P^a_n \ge P^a_{n-1}\}} q^a_{n-1}$$

and $\mathrm{OFI}_k = \sum_{n \in (t_{k-1}, t_k]} e_n$. Intuition: order flow that adds bid depth or
removes ask depth pushes the price up, and vice versa, regardless of whether it arrives as a
limit order, a cancel, or a trade.

**Data notes.** Prices are in $10^{-4}$ dollars, so one tick ($0.01) = 100 units. Rows with a
one-sided book (pre-open build-up) are dropped, and the sample is restricted to 09:30–16:00.
Intervals with no BBO events don't appear (the paper drops these too); the mid change then spans
the gap.

In [1]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from analysis.ofi import TICK, DT_NS, event_order_flow, fit_price_impact, load_bbo, ofi_intervals, window_fits

TICKERS = ["SPY", "QQQ", "AMD", "INTC", "GOOGL"]
COLOR = dict(zip(TICKERS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))

events = {t: event_order_flow(load_bbo(t)) for t in TICKERS}  # the expensive part, computed once
intervals = {t: ofi_intervals(ev) for t, ev in events.items()}
fits = {t: fit_price_impact(iv) for t, iv in intervals.items()}

## Results: full-day fit per ticker

Test criteria: $\beta > 0$ with $t(\beta) > 2$; $R^2$ compared against the paper's ~65% average.
$\beta$ is in ticks per share of imbalance.

In [2]:
summary = pl.DataFrame(
    [
        {
            "ticker": t,
            "events": len(events[t]),
            "intervals": len(intervals[t]),
            "beta_ticks_per_share": f.params[1],
            "t_beta": f.tvalues[1],
            "r2": f.rsquared,
        }
        for t, f in fits.items()
    ]
)
summary

ticker,events,intervals,beta_ticks_per_share,t_beta,r2
str,i64,i64,f64,f64,f64
"""SPY""",4161382,2339,0.000141,23.659311,0.3337
"""QQQ""",4207542,2339,0.000194,24.64568,0.609387
"""AMD""",2341809,2339,0.000141,39.924305,0.789099
"""INTC""",1586139,2339,0.000228,5.743945,0.635069
"""GOOGL""",1487675,2339,0.009397,10.482929,0.137655


## Price impact: $\Delta P$ vs OFI

Left-to-right per ticker: 2 000 randomly sampled 10s intervals with the fitted line (the paper's
Fig. 1 view), then the linearity check: mean $\Delta P$ within 50 OFI-quantile bins. A linear
relation puts the binned means on a straight line through the origin; a concave S-shape would
mimic the paper's *trade imbalance* result and count against the hypothesis.

In [3]:
fig = make_subplots(
    rows=len(TICKERS), cols=2, column_titles=["10s intervals (2 000 sampled) + OLS fit", "mean dP in 50 OFI-quantile bins"],
    row_titles=TICKERS, horizontal_spacing=0.08, vertical_spacing=0.04,
)
for r, t in enumerate(TICKERS, start=1):
    iv, f = intervals[t], fits[t]
    smp = iv.sample(n=min(2000, len(iv)), seed=7)
    fig.add_scatter(
        x=smp["ofi"], y=smp["dP"], mode="markers",
        marker=dict(color=COLOR[t], size=4, opacity=0.35), row=r, col=1, showlegend=False,
    )
    xs = [float(iv["ofi"].min()), float(iv["ofi"].max())]
    fig.add_scatter(
        x=xs, y=[f.params[0] + f.params[1] * x for x in xs], mode="lines",
        line=dict(color="#444444", width=2), row=r, col=1, showlegend=False,
    )
    binned = (
        iv.with_columns(pl.col("ofi").qcut(50, labels=[str(i) for i in range(50)]).alias("bin"))
        .group_by("bin")
        .agg(pl.col("ofi").mean().alias("ofi_mean"), pl.col("dP").mean().alias("dP_mean"))
        .sort("ofi_mean")
    )
    fig.add_scatter(
        x=binned["ofi_mean"], y=binned["dP_mean"], mode="markers",
        marker=dict(color=COLOR[t], size=7), row=r, col=2, showlegend=False,
    )
fig.update_layout(height=1500, width=950, template="plotly_white", title="Mid-price change (ticks) vs OFI (shares)")
fig.show()

## Robustness: interval length

Re-bucket the same event streams at $\Delta t \in \{1, 5, 10, 30, 60\}$ s and refit. $R^2$ shares
one axis; $\beta$ gets small multiples because its scale is per-ticker (GOOGL's top-of-book depth
is ~100× thinner than SPY's, so its $\beta$ is ~100× larger).

In [4]:
DTS = [1, 5, 10, 30, 60]
sweep = pl.DataFrame(
    [
        {"ticker": t, "dt_s": dt, "beta": f.params[1], "t_beta": f.tvalues[1], "r2": f.rsquared}
        for t, ev in events.items()
        for dt in DTS
        for f in [fit_price_impact(ofi_intervals(ev, dt * 10**9))]
    ]
)

display(sweep.pivot(on="ticker", index="dt_s", values="t_beta"))  # t(beta) at each interval length

fig = make_subplots(rows=1, cols=2, subplot_titles=["R² vs Δt", "β vs Δt (per-ticker scale)"])
for t in TICKERS:
    s = sweep.filter(pl.col("ticker") == t).sort("dt_s")
    fig.add_scatter(x=s["dt_s"], y=s["r2"], mode="lines+markers", name=t,
                    line=dict(color=COLOR[t], width=2), marker=dict(size=8), row=1, col=1)
    fig.add_scatter(x=s["dt_s"], y=s["beta"] / s["beta"][s["dt_s"].to_list().index(10)],
                    mode="lines+markers", showlegend=False,
                    line=dict(color=COLOR[t], width=2), marker=dict(size=8), row=1, col=2)
fig.update_xaxes(type="log", title_text="Δt (s)")
fig.update_yaxes(title_text="R²", row=1, col=1)
fig.update_yaxes(title_text="β / β(10s)", row=1, col=2)
fig.update_layout(height=420, width=950, template="plotly_white",
                  title="Interval-length sweep (β normalised by its 10s value)")
fig.show()

dt_s,SPY,QQQ,AMD,INTC,GOOGL
i64,f64,f64,f64,f64,f64
1,63.760689,101.270945,42.760871,11.296,13.967697
5,31.111882,36.257723,38.271966,5.40483,12.555289
10,23.659311,24.64568,39.924305,5.743945,10.482929
30,7.359773,11.494451,32.387411,5.309681,6.01211
60,6.240896,9.181491,29.698347,6.505586,4.893282


## Intraday stability of $\beta$

Refit per half-hour window ($\pm 2$ standard errors). The paper estimates within half-hour windows
precisely because depth varies intraday and $\beta$ moves inversely with depth, so expect larger
$\beta$ near the open when the book is thin. Small multiples, per-ticker scale.

In [5]:
fig = make_subplots(rows=len(TICKERS), cols=1, row_titles=TICKERS, vertical_spacing=0.05,
                    shared_xaxes=True)
for r, t in enumerate(TICKERS, start=1):
    c = COLOR[t]
    band = f"rgba({int(c[1:3], 16)},{int(c[3:5], 16)},{int(c[5:7], 16)},0.2)"
    wf = window_fits(events[t])
    x = [w * 0.5 - 9.5 + 0.25 for w in wf["window"]]  # window centre, hours after 09:30
    fig.add_scatter(x=x, y=(wf["beta"] + 2 * wf["se"]).to_list(), mode="lines",
                    line=dict(width=0), showlegend=False, row=r, col=1)
    fig.add_scatter(x=x, y=(wf["beta"] - 2 * wf["se"]).to_list(), mode="lines", line=dict(width=0),
                    fill="tonexty", fillcolor=band, showlegend=False, row=r, col=1)
    fig.add_scatter(x=x, y=wf["beta"].to_list(), mode="lines+markers",
                    line=dict(color=COLOR[t], width=2), marker=dict(size=7), showlegend=False, row=r, col=1)
    fig.update_yaxes(rangemode="tozero", row=r, col=1)
fig.update_xaxes(title_text="hours after 09:30", row=len(TICKERS), col=1)
fig.update_layout(height=1100, width=950, template="plotly_white",
                  title="β per half-hour window ± 2 s.e. (ticks/share, per-ticker scale)")
fig.show()

## Does the model track the price? (AMD, 20 minutes)

Cumulative actual mid-price change vs the model's prediction $\beta \cdot \mathrm{cumulative \space OFI}$
(both in ticks, one axis) over a 20-minute window.

In [6]:
t0 = 14 * 3600 * 10**9  # 14:00
seg = (
    intervals["AMD"]
    .filter((pl.col("bucket") * DT_NS >= t0) & (pl.col("bucket") * DT_NS < t0 + 1200 * 10**9))
    .with_columns(
        pl.col("dP").cum_sum().alias("actual"),
        (pl.col("ofi").cum_sum() * fits["AMD"].params[1]).alias("predicted"),
    )
)
x = [(b * 10 - 14 * 3600) / 60 for b in seg["bucket"]]
fig = go.Figure()
fig.add_scatter(x=x, y=seg["actual"], mode="lines", name="actual Δmid",
                line=dict(color=COLOR["AMD"], width=2))
fig.add_scatter(x=x, y=seg["predicted"], mode="lines", name="β · cum. OFI",
                line=dict(color="#4a3aa7", width=2, dash="dot"))
fig.update_layout(height=380, width=950, template="plotly_white",
                  title="AMD 14:00–14:20: cumulative mid change vs OFI prediction (ticks)",
                  xaxis_title="minutes after 14:00", yaxis_title="ticks")
fig.show()

## Book shape

Average mid-price, quoted spread (in ticks and in basis points of the mid), and size at the
best quotes, for interpreting the differences in $R^2$ between tickers. The tick is a fixed
\$0.01, so high-priced names quote spreads that are wide in ticks even when the relative
spread is ordinary.

In [7]:
book_shape = pl.DataFrame(
    [
        {
            "ticker": t,
            "avg_mid_usd": ev.select((pl.col("mid") / 10_000).mean()).item(),
            "avg_spread_ticks": ev.select(((pl.col("ask_px_00") - pl.col("bid_px_00")) / TICK).mean()).item(),
            "avg_spread_bps": ev.select(((pl.col("ask_px_00") - pl.col("bid_px_00")) / pl.col("mid") * 10_000).mean()).item(),
            "avg_shares_at_best": ev.select(((pl.col("bid_sz_00") + pl.col("ask_sz_00")) / 2).mean()).item(),
        }
        for t, ev in events.items()
    ]
)
book_shape

ticker,avg_mid_usd,avg_spread_ticks,avg_spread_bps,avg_shares_at_best
str,f64,f64,f64,f64
"""SPY""",325.179753,1.433131,0.440724,793.84959
"""QQQ""",220.966889,1.256931,0.568845,1024.139479
"""AMD""",47.371718,1.128563,2.382841,1673.866835
"""INTC""",65.743631,1.133787,1.724485,1025.960427
"""GOOGL""",1442.908586,64.167176,4.446839,43.935497


## Conclusion

The hypothesis is supported for the small-spread, deep-book names, and only partially overall.

| ticker | β (ticks/share) | t(β) | R² | verdict |
|---|---|---|---|---|
| AMD | 1.41e-4 | 39.9 | 0.79 | supported |
| INTC | 2.28e-4 | 5.7 | 0.64 | supported |
| QQQ | 1.94e-4 | 24.6 | 0.61 | supported |
| SPY | 1.41e-4 | 23.7 | 0.33 | partially (β robust, R² below paper's range) |
| GOOGL | 9.40e-3 | 10.5 | 0.14 | not supported on R² |

- Every β is positive and strongly significant, and the binned means fall on a straight line
  through the origin, so the sign and linearity of the impact replicate for all five names.
- R² reaches the paper's 50-70% range only for the names that quote a spread near one tick with
  large size at the touch (book shape table above). GOOGL quotes a spread of tens of ticks with
  very little size at the best quotes, so most price formation happens beyond level 1 and
  top-of-book OFI explains little.
- The result is not specific to the 10s interval. The sweep shows a significant linear relation
  at every Δt from 1s to 60s, with R² increasing in Δt for AMD and INTC and decreasing for SPY
  and QQQ.
- The half-hour fits show β is not constant over the day, with GOOGL's falling sharply after the
  open. This is consistent with the paper's inverse depth dependence β ≈ c/AD^λ, tested in
  `depth-analysis.ipynb`.

Caveats: one trading day, five tickers, level-1 depth only, empty 10s buckets dropped, and
White (HC1) standard errors, robust to heteroskedasticity but not to serial correlation.